"""
STEP 1 — SCRIPT OVERVIEW (Energy Results Totals Update)

This script iterates through all building folders inside:
    /Users/jakegehrung/Desktop/data_raw/baseline_models

Each building folder is named using its BUILDING_REFERENCE_NUMBER and contains:
    TOTAL/energy_results.csv

For each energy_results.csv file, the script recalculates and overwrites the following columns:

- electricity_tot_1
- lpg_tot_1
- mineral_wood_tot_1
- mains_gas_tot_1
- oil_tot_1
- wood_logs_tot_1
- wood_pellets_tot_1
- wood_chips_tot_1
- coal_tot_1

The calculations are:

electricity_tot_1 = electricity_heat + electricity_hot_water + electricity_app_light_vent - existing_pv
lpg_tot_1 = lpg_heat + lpg_hot_water
mineral_wood_tot_1 = mineral_wood_heat + mineral_wood_hot_water
mains_gas_tot_1 = mains_gas_heat + mains_gas_hot_water
oil_tot_1 = oil_heat + oil_hot_water
wood_logs_tot_1 = wood_logs_heat + wood_logs_hot_water
wood_pellets_tot_1 = wood_pellets_heat + wood_pellets_hot_water
wood_chips_tot_1 = wood_chips_heat + wood_chips_hot_water
coal_tot_1 = coal_heat + coal_hot_water

All missing values are treated as 0 to avoid calculation errors.

The script overwrites each energy_results.csv in place.
"""

In [1]:
# STEP 2 — IMPORTS & BASE DIRECTORY SETUP

import os
from pathlib import Path
import pandas as pd

# Base directory containing all building folders
BASE_DIR = Path("/Users/jakegehrung/Desktop/data_raw/baseline_models")

# Optional: EPC dataset (not required for computation, but kept for reference)
EPC_PATH = Path("/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_with_fuel_codes.csv")

# Load EPC (optional use / validation)
epc_df = pd.read_csv(EPC_PATH)

print(f"Loaded EPC dataset with {len(epc_df)} rows")
print(f"Scanning building folders in: {BASE_DIR}")

Loaded EPC dataset with 117 rows
Scanning building folders in: /Users/jakegehrung/Desktop/data_raw/baseline_models


In [2]:
# STEP 3 — DEFINE ENERGY CALCULATION FUNCTION

def compute_energy_totals(df: pd.DataFrame) -> pd.DataFrame:
    """
    Takes an energy_results dataframe and computes total fuel columns.
    """

    # Fill NaNs with 0 for safe arithmetic
    df = df.fillna(0)

    # Electricity total
    df["electricity_tot_1"] = (
        df["electricity_heat"]
        + df["electricity_hot_water"]
        + df["electricity_app_light_vent"]
        - df["existing_pv"]
    )

    # Gas and fuels
    df["lpg_tot_1"] = df["lpg_heat"] + df["lpg_hot_water"]
    df["mineral_wood_tot_1"] = df["mineral_wood_heat"] + df["mineral_wood_hot_water"]
    df["mains_gas_tot_1"] = df["mains_gas_heat"] + df["mains_gas_hot_water"]
    df["oil_tot_1"] = df["oil_heat"] + df["oil_hot_water"]

    # Biomass fuels
    df["wood_logs_tot_1"] = df["wood_logs_heat"] + df["wood_logs_hot_water"]
    df["wood_pellets_tot_1"] = df["wood_pellets_heat"] + df["wood_pellets_hot_water"]
    df["wood_chips_tot_1"] = df["wood_chips_heat"] + df["wood_chips_hot_water"]

    # Solid fuel
    df["coal_tot_1"] = df["coal_heat"] + df["coal_hot_water"]

    return df

In [3]:
# STEP 4 — LOOP THROUGH BUILDINGS AND UPDATE FILES

building_folders = [f for f in BASE_DIR.iterdir() if f.is_dir()]

print(f"Found {len(building_folders)} building folders")

updated_count = 0
missing_count = 0

for folder in building_folders:
    building_id = folder.name
    energy_file = folder / "TOTAL" / "energy_results.csv"

    if not energy_file.exists():
        print(f"[MISSING] {building_id}: energy_results.csv not found")
        missing_count += 1
        continue

    try:
        df = pd.read_csv(energy_file)

        df_updated = compute_energy_totals(df)

        df_updated.to_csv(energy_file, index=False)

        updated_count += 1

    except Exception as e:
        print(f"[ERROR] {building_id}: {e}")

print("\nDONE")
print(f"Updated files: {updated_count}")
print(f"Missing files: {missing_count}")

Found 117 building folders

DONE
Updated files: 117
Missing files: 0
